In [1]:
import ydf
import pandas as pd
import numpy as np

In [2]:
crime_dataset = pd.read_csv("./Crime_Data_from_2020_to_2024.csv")
crime_dataset = crime_dataset.drop(columns = ["DR_NO", "Date Rptd", "AREA", "Part 1-2", "Crm Cd", "Mocodes", "Premis Cd", "Weapon Used Cd", "Status Desc", "LOCATION", "Cross Street"])

crime_dataset["DATE OCC"] = pd.to_datetime(crime_dataset["DATE OCC"], format = "%Y %b %d %I:%M:%S %p")
crime_dataset = crime_dataset[crime_dataset["DATE OCC"].dt.year == 2020]
crime_dataset["DATE OCC"] = crime_dataset["DATE OCC"].dt.day_of_year

crime_dataset.loc[crime_dataset["Status"] == "AA", "Status"] = "IR"
crime_dataset.loc[crime_dataset["Status"] == "AO", "Status"] = "IR"
crime_dataset.loc[crime_dataset["Status"] == "JA", "Status"] = "IR"
crime_dataset.loc[crime_dataset["Status"] == "JO", "Status"] = "IR"
crime_dataset.head(20)



,DATE OCC,TIME OCC,AREA NAME,Rpt Dist No,Crm Cd Desc,Vict Age,Vict Sex,Vict Descent,Premis Desc,Weapon Desc,Status,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LAT,LON
0,312,845,N Hollywood,1502,THEFT OF IDENTITY,31,M,H,SINGLE FAMILY DWELLING,NaN,IC,354.0,NaN,NaN,NaN,34.2124,-118.4092
1,292,1845,N Hollywood,1521,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",32,M,H,SIDEWALK,KNIFE WITH BLADE 6INCHES OR LESS,IC,230.0,NaN,NaN,NaN,34.1993,-118.4203
2,304,1240,Van Nuys,933,THEFT OF IDENTITY,30,M,W,SINGLE FAMILY DWELLING,NaN,IC,354.0,NaN,NaN,NaN,34.1847,-118.4509
3,359,1310,Wilshire,782,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,47,F,A,STREET,NaN,IC,331.0,NaN,NaN,NaN,34.0339,-118.3747
4,273,1830,Pacific,1454,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),63,M,H,ALLEY,NaN,IC,420.0,NaN,NaN,NaN,33.9813,-118.4350
5,316,1210,Hollenbeck,429,THEFT OF IDENTITY,35,M,B,"MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)",NaN,IC,354.0,NaN,NaN,NaN,34.0830,-118.1678
6,107,1350,Southwest,396,THEFT OF IDENTITY,21,F,B,SINGLE FAMILY DWELLING,NaN,IC,354.0,NaN,NaN,NaN,34.0100,-118.2900
7,189,1400,Northeast,1133,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,14,F,H,YARD (RESIDENTIAL/BUSINESS),UNKNOWN WEAPON/OTHER WEAPON,IR,812.0,860.0,NaN,NaN,34.1107,-118.2589
8,62,1200,Devonshire,1729,THEFT OF IDENTITY,43,M,W,SINGLE FAMILY DWELLING,NaN,IC,354.0,NaN,NaN,NaN,34.2763,-118.5210
9,245,900,Topanga,2196,THEFT OF IDENTITY,57,M,W,SINGLE FAMILY DWELLING,NaN,IC,354.0,NaN,NaN,NaN,34.1493,-118.5886


In [3]:
# Set missing values to unknown (X)
crime_dataset['Vict Sex'] = crime_dataset['Vict Sex'].replace(np.nan, "X")
crime_dataset['Vict Sex'] = crime_dataset['Vict Sex'].replace("-", "X")
crime_dataset['Vict Sex'] = crime_dataset['Vict Sex'].replace("H", "X")

crime_dataset['Vict Descent'] = crime_dataset['Vict Descent'].replace(np.nan, "X")
crime_dataset['Vict Descent'] = crime_dataset['Vict Descent'].replace("-", "X")

In [4]:
# Remove negative ages
crime_dataset.loc[crime_dataset['Vict Age'] < 0, 'Vict Age'] = np.nan

In [5]:
#Force strings
crime_dataset["Premis Desc"] = (crime_dataset["Premis Desc"].fillna("UNKNOWN").apply(lambda x: str(x)))
crime_dataset["Weapon Desc"] = (crime_dataset["Weapon Desc"].fillna("UNKNOWN").apply(lambda x: str(x)))

In [6]:
def split_dataset(ds, test_ratio):
  test_indices = np.random.rand(len(ds)) < test_ratio
  return ds[~test_indices], ds[test_indices]

crime_train, crime_test = split_dataset(crime_dataset, 0.10)
print(len(crime_train), len(crime_test))

179960 19887


In [7]:
crime_IC = crime_train[crime_train["Status"] == "IC"]
crime_CC = crime_train[crime_train["Status"] == "CC"]
crime_IR = crime_train[crime_train["Status"] == "IR"]

print(len(crime_IC), len(crime_CC), len(crime_IR))

136779 0 43181


In [8]:
crime_IC_keep, _ = split_dataset(crime_IC, 0.3) # keep 70% of IC
crime_IR_dup = pd.concat([crime_IR] * 2)  # duplicate IR entries
print(len(crime_IC_keep), len(crime_IR_dup))

95705 86362


In [9]:
crime_resample = pd.concat([crime_IC_keep, crime_IR_dup])
crime_resample = crime_resample.sample(frac = 1).reset_index(drop=True) # reshuffle
crime_resample.head(20)

,DATE OCC,TIME OCC,AREA NAME,Rpt Dist No,Crm Cd Desc,Vict Age,Vict Sex,Vict Descent,Premis Desc,Weapon Desc,Status,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LAT,LON
0,328,1310,77th Street,1253,VIOLATION OF RESTRAINING ORDER,43.0,F,B,SINGLE FAMILY DWELLING,UNKNOWN,IR,901.0,NaN,NaN,NaN,33.9709,-118.3112
1,44,1230,Northeast,1109,ROBBERY,0.0,X,X,MOTEL,VERBAL THREAT,IC,210.0,998.0,NaN,NaN,34.1439,-118.1955
2,82,40,Harbor,566,"VANDALISM - FELONY ($400 & OVER, ALL CHURCH VA...",38.0,M,H,"VEHICLE, PASSENGER/TRUCK",UNKNOWN,IR,740.0,NaN,NaN,NaN,33.7360,-118.2835
3,329,1740,West LA,841,THEFT PLAIN - PETTY ($950 & UNDER),33.0,F,W,OTHER RESIDENCE,UNKNOWN,IC,440.0,NaN,NaN,NaN,34.0425,-118.4656
4,219,1700,Northeast,1138,BURGLARY,0.0,X,X,LIBRARY,UNKNOWN,IR,310.0,NaN,NaN,NaN,34.1114,-118.1891
5,239,600,Southeast,1822,ATTEMPTED ROBBERY,0.0,X,X,GAS STATION,KNIFE WITH BLADE 6INCHES OR LESS,IR,220.0,998.0,NaN,NaN,33.9462,-118.2739
6,27,2013,Central,124,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",62.0,M,H,SIDEWALK,UNKNOWN WEAPON/OTHER WEAPON,IR,230.0,NaN,NaN,NaN,34.0530,-118.2444
7,139,735,Harbor,514,VIOLATION OF TEMPORARY RESTRAINING ORDER,31.0,F,H,SINGLE FAMILY DWELLING,UNKNOWN,IR,902.0,NaN,NaN,NaN,33.7977,-118.2773
8,195,1200,Olympic,2093,BATTERY WITH SEXUAL CONTACT,31.0,F,B,NURSING/CONVALESCENT/RETIREMENT HOME,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",IR,860.0,NaN,NaN,NaN,34.0400,-118.3029
9,273,530,Wilshire,742,"VANDALISM - FELONY ($400 & OVER, ALL CHURCH VA...",25.0,M,W,"VEHICLE, PASSENGER/TRUCK",UNKNOWN,IC,740.0,NaN,NaN,NaN,34.0599,-118.3698


In [10]:
crime_resample.value_counts("Status")

Status
IC    95705
IR    86362
Name: count, dtype: int64

In [11]:
model = ydf.RandomForestLearner(label = "Status", num_discretized_numerical_bins = 510).train(crime_resample)

Train model on 182067 examples
Model trained in 0:00:08.324254


In [12]:
model.evaluate(crime_test)

Label \ Pred,IC,IR
IC,12439,1316
IR,2689,3443


In [ ]:
#model.plot_tree(tree_idx=0, max_depth= 5)

In [13]:
model.analyze(crime_test)